# I - Khai báo thư viện

In [1]:
import pandas as pd
import numpy as np
import torch
from scipy.sparse import csr_matrix
from tqdm.auto import tqdm
import math
from IPython.display import HTML, display, display_html

c:\Users\NGUYEN GIA KHANH\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# II - Chọn thiết bị huấn luyện mô hình

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


# III - Đọc dữ liệu

### 1 - Đọc dữ liệu từ file csv

In [3]:
df = pd.read_csv("./data/data-commerce-clean.csv")
df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00 UTC,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
1,2019-10-01 00:00:01 UTC,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
2,2019-10-01 00:00:04 UTC,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d
3,2019-10-01 00:00:05 UTC,view,1480613,2053013561092866779,computers.desktop,pulser,908.62,512742880,0d0d91c2-c9c2-4e81-90a5-86594dec0db9
4,2019-10-01 00:00:10 UTC,view,28719074,2053013565480109009,apparel.shoes.keds,baden,102.71,520571932,ac1cd4e5-a3ce-4224-a2d7-ff660a105880


### 2 - Tiền xử lý dữ liệu và mã hóa người dùng – sản phẩm cho mô hình gợi ý : 
#### 2.1 Chuyển loại sự kiện thành điểm tương tác (event strength) : 
- Mỗi loại hành động của người dùng được gán một mức độ quan trọng khác nhau 
- Mã hóa user_id và product_id thành mã số liên tục (categorical encoding) 
- Đếm số lượng người dùng và sản phẩm

In [4]:
df["event_strength"] = (
    df["event_type"].map({"view": 1, "cart": 3, "purchase": 5}).fillna(0)
)

df["UserID"] = df["user_id"].astype("category").cat.codes
df["ProductID"] = df["product_id"].astype("category").cat.codes

num_users = df["UserID"].nunique()
num_items = df["ProductID"].nunique()

print("Users:", num_users, "Items:", num_items)

Users: 2323036 Items: 60371


#### 2.2 Tạo ma trận tương tác thưa (Sparse User–Item Matrix) cho mô hình gợi ý 
- Gom nhóm (groupby) theo UserID và ProductID 
- Tạo 3 mảng rows, cols, vals 
- Xây dựng ma trận thưa bằng CSR Matrix (Compressed Sparse Row)

In [5]:
grouped = df.groupby(["UserID", "ProductID"])["event_strength"].sum().reset_index()

rows = grouped["UserID"].values
cols = grouped["ProductID"].values
vals = grouped["event_strength"].astype(float).values

sparse_user_item = csr_matrix((vals, (rows, cols)), shape=(num_users, num_items))

print("Sparse shape:", sparse_user_item.shape, "nnz:", sparse_user_item.nnz)

Sparse shape: (2323036, 60371) nnz: 13641671


#### 2.3 Tạo bảng thông tin sản phẩm (Item Metadata Table)

In [6]:
items_df = (
    df[["ProductID", "product_id", "category_code", "brand"]]
    .drop_duplicates()
    .set_index("ProductID")
)

items_df.head()

,product_id,category_code,brand
ProductID,,,
10939,3900821,appliances.environment.water_heater,aqua
2438,1307067,computers.notebook,lenovo
522,1004237,electronics.smartphone,apple
3237,1480613,computers.desktop,pulser
51686,28719074,apparel.shoes.keds,baden


# IV - Mô hình Alternating Least Squares ( ALS )

### 4.1 Xây dựng mô hình ALS bằng PyTorch (ALS_PyTorch Class)

In [7]:
class ALS_PyTorch:
    def __init__(
        self,
        num_users,
        num_items,
        factors=40,
        reg=0.01,
        iterations=10,
        alpha=40.0,
        seed=42,
    ):
        self.num_users = int(num_users)
        self.num_items = int(num_items)
        self.factors = factors
        self.reg = reg
        self.iterations = iterations
        self.alpha = alpha

        rng = np.random.RandomState(seed)
        self.U = torch.tensor(
            0.01 * rng.randn(num_users, factors), dtype=torch.float32, device=device
        )
        self.V = torch.tensor(
            0.01 * rng.randn(num_items, factors), dtype=torch.float32, device=device
        )

    def fit(self, R: csr_matrix):
        indptr = R.indptr
        indices = R.indices
        data = R.data

        regI = self.reg * torch.eye(self.factors, device=device)

        for it in range(self.iterations):
            print(f"\nIteration {it+1}/{self.iterations}: ")

            V = self.V
            VtV = V.T @ V

            for u in tqdm(range(self.num_users), desc="Updating users"):
                start = indptr[u]
                end = indptr[u + 1]
                if start == end:
                    continue

                item_idx = indices[start:end]
                r_ui = torch.tensor(data[start:end], dtype=torch.float32, device=device)

                Y = V[item_idx] 
                w = self.alpha * r_ui 

                A = VtV + (Y.T * w) @ Y + regI
                b = Y.T @ (1 + w)

                self.U[u] = torch.linalg.solve(A, b)

            R_csc = R.tocsc()
            c_indptr = R_csc.indptr
            c_indices = R_csc.indices
            c_data = R_csc.data

            X = self.U
            XtX = X.T @ X

            for i in tqdm(range(self.num_items), desc="Updating items"):
                start = c_indptr[i]
                end = c_indptr[i + 1]
                if start == end:
                    continue

                user_idx = c_indices[start:end]
                r_ui = torch.tensor(
                    c_data[start:end], dtype=torch.float32, device=device
                )

                X_nz = X[user_idx]
                w = self.alpha * r_ui

                A = XtX + (X_nz.T * w) @ X_nz + regI
                b = X_nz.T @ (1 + w)

                self.V[i] = torch.linalg.solve(A, b)

        print("Finish training ALS.")
        return self

    def recommend(self, user_idx, R, N=20):
        
        user_vec = self.U[user_idx]
        scores = (self.V @ user_vec).detach().cpu().numpy()

        seen = R.getrow(user_idx).indices
        scores[seen] = -np.inf

        top = np.argpartition(-scores, N)[:N]
        top = top[np.argsort(-scores[top])]

        return top, scores[top]

### 4.2 Huấn Luyện Mô Hình

In [9]:
als = ALS_PyTorch(
    num_users=num_users,
    num_items=num_items,
    factors=64,
    reg=0.05,
    iterations=5,
    alpha=40,
)

als.fit(sparse_user_item)


Iteration 1/5: 


Updating items: 100%|██████████| 60371/60371 [00:58<00:00, 1023.85it/s]



Iteration 2/5: 


Updating items: 100%|██████████| 60371/60371 [00:35<00:00, 1713.56it/s]



Iteration 3/5: 


...


Iteration 4/5: 


Updating items: 100%|██████████| 60371/60371 [00:31<00:00, 1921.06it/s]



Iteration 5/5: 


Updating items: 100%|██████████| 60371/60371 [00:50<00:00, 1197.85it/s]

Finish training ALS.


### 4.4 Lưu Mô Hình Huấn Luyện

In [14]:
def save_als_model(model, path="als_pytorch_model.pth"):
    save_dict = {
        "num_users": model.num_users,
        "num_items": model.num_items,
        "factors": model.factors,
        "reg": model.reg,
        "iterations": model.iterations,
        "alpha": model.alpha,
        "U": model.U.detach().cpu(),
        "V": model.V.detach().cpu(),
    }
    torch.save(save_dict, path)
    print(f"Saved ALS model to {path}")
    
save_als_model(als, "als_pytorch_model.pth")

Saved ALS model to als_pytorch_model.pth


### 4.5 Load lại mô hình

In [ ]:
def load_als_model(path="./models/als_pytorch_model.pth", device="cpu"):
    data = torch.load(path, map_location=device)

    model = ALS_PyTorch(
        num_users=data["num_users"],
        num_items=data["num_items"],
        factors=data["factors"],
        reg=data["reg"],
        iterations=data["iterations"],
        alpha=data["alpha"],
    )

    model.U = data["U"].to(device)
    model.V = data["V"].to(device)

    print(f"Loaded ALS model from {path}")
    return model

# V - Hàm tạo gợi ý cho người dùng và hiển thị lịch sử – đề xuất song song

In [9]:
def user_recommend(model, items_df, grouped_df, R, user_id='rand', n=20, fill_na=True):

    if user_id == 'rand':
        user_id = int(grouped_df["UserID"].sample(1).values[0])

    if user_id not in grouped_df["UserID"].values:
        raise ValueError(f"UserID {user_id} không tồn tại trong dữ liệu!")

    history = grouped_df[grouped_df["UserID"] == user_id][
        ["ProductID", "event_strength"]
    ].merge(items_df, left_on="ProductID", right_index=True)

    if fill_na:
        history["category_code"] = history["category_code"].fillna("N/A")
        history["brand"] = history["brand"].fillna("N/A")

    history_grouped = (
        history.groupby(["category_code", "brand"])["event_strength"].sum()
        .sort_values(ascending=False)
    )

    top_items, _ = model.recommend(user_id, R, N=n)
    recommended = items_df.loc[top_items].copy()

    if fill_na:
        recommended["category_code"] = recommended["category_code"].fillna("N/A")
        recommended["brand"] = recommended["brand"].fillna("N/A")

    recommend_grouped = (
        recommended.groupby(["category_code", "brand"])["product_id"].count()
        .sort_values(ascending=False)
    )

    hist_styler = history_grouped.to_frame(name="event_strength").style.set_table_attributes(
        "style='display:inline'"
    )
    rec_styler = recommend_grouped.to_frame(name="count").style.set_table_attributes(
        "style='display:inline'"
    )

    display(HTML(f"<h2>User ID: {user_id}</h2>"))
    display(HTML("<h2>User History" + "&nbsp;" * 50 + "Recommendations</h2>"))
    display_html(
        hist_styler._repr_html_()
        + "&nbsp;" * 50
        + rec_styler._repr_html_(),
        raw=True
    )

    return top_items

#### 5.1 Tải mô hình ALS đã huấn luyện và tạo gợi ý cho một người dùng ngẫu nhiên

In [ ]:
als_loaded = load_als_model("./models/als_pytorch_model.pth", device)

user_recommend(
    model=als_loaded,
    items_df=items_df,
    grouped_df=grouped,
    R=sparse_user_item,
    user_id='rand',
    n=20
)

C:\Users\NGUYEN GIA KHANH\AppData\Local\Temp\ipykernel_24140\3826918746.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(path, map_location=device)


Loaded ALS model from als_pytorch_model.pth


event_strength 
 
 
 category_code 
 brand 
   
 
 
 
 
 appliances.environment.vacuum 
 tefal 
 5 
 
 
 bosch 
 4 
 
 
 vitek 
 3 
 
 
 appliances.sewing_machine 
 chayka 
 2 
 
 
 appliances.kitchen.microwave 
 dauscher 
 2 
 
 
 appliances.kitchen.coffee_machine 
 jura 
 2 
 
 
 appliances.environment.vacuum 
 karcher 
 1 
 
 
 philips 
 1 
 
 
 samsung 
 1 
 
 
 appliances.sewing_machine 
 janome 
 1 
 
 
 furniture.living_room.cabinet 
 brw 
 1 
 
 
 
                                                  
 
 
 
   
   
 count 
 
 
 category_code 
 brand 
   
 
 
 
 
 appliances.environment.vacuum 
 samsung 
 11 
 
 
 bosch 
 3 
 
 
 lg 
 2 
 
 
 elenberg 
 1 
 
 
 tefal 
 1 
 
 
 appliances.kitchen.microwave 
 arg 
 1 
 
 
 elenberg 
 1

array([ 9909,  9831,  9717, 10086,  9971,  9818,  9995, 10043,  8346,
        9729,  9903, 10045,  9739,  9792,  9709,  9884,  9825,  9826,
        8578,  9836])